# Lesson 5: Human in the Loop

### 本节课的核心思路：Human in the Loop（人工干预）

在真实场景里，你往往不希望 Agent 完全自主执行敏感操作（比如发邮件、下单、调用有副作用的工具），
而是希望在关键节点暂停下来，让人先看一眼、甚至修改一下参数，再决定要不要继续。
这一课依托 Lesson 4 学到的 **持久化（checkpointer）** 能力，介绍几个新概念：

- **`interrupt_before` / `interrupt_after`**：编译图时指定"在执行某个节点之前/之后暂停"，
  暂停后图会把当前 state 存进 checkpointer 并直接返回，不会继续往下跑，直到你再次 `stream(None, thread)` 才会继续。
- **`get_state(thread)`**：查看某个 thread 当前暂停在哪个 state、下一步 (`.next`) 该跑哪个节点。
- **`update_state(thread, values)`**：在暂停点手动修改 state（比如改掉模型想要调用的工具参数），再继续执行——这就是"人工审核/修改后再放行"。
- **`get_state_history(thread)` + 时间旅行（Time Travel）**：checkpointer 会保留每一步的历史快照，
  你可以取出任意一个历史快照的 `config`，把它当成新的起点重新 `stream`，从而"回到过去"重新分叉出一条不同的执行路径。

这一课后半部分换了一个更简单的 `node1/node2` 计数器图（不涉及 LLM），专门用来更清晰地演示 state history、时间旅行、以及手动修改历史 state 后再 replay 的行为。

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults  # 旧版路径，当前仍可用（新版推荐 langchain_tavily.TavilySearch）
from langgraph.checkpoint.sqlite import SqliteSaver

# 【版本变化修复】和 Lesson 4 一样：新版 langgraph 里 SqliteSaver.from_conn_string(":memory:") 返回的是
# 一个上下文管理器，不是可以直接用的 SqliteSaver 实例，直接赋值给 memory 会导致后面读写检查点时报
# AttributeError: '_GeneratorContextManager' object has no attribute 'get_tuple'。
# 用 .__enter__() 手动进入上下文拿到真正的实例；同时把上下文管理器对象保存到 _memory_cm，
# 避免被垃圾回收提前关闭底层 sqlite 连接（否则后面会报 "Cannot operate on a closed database."）。
_memory_cm = SqliteSaver.from_conn_string(":memory:")
memory = _memory_cm.__enter__()

In [ ]:
from uuid import uuid4
from langchain_core.messages import AnyMessage,SystemMessage,HumanMessage,AIMessage


"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""

def reduce_messages(left:list[AnyMessage],right:list[AnyMessage])->list[AnyMessage]:
    # 给 right 里没有 id 的消息生成一个随机 id（新产生的消息一般没有 id）
    for message in right:
        if not message.id:
            message.id=str(uuid4())
    merged=left.copy()
    for message in right:
        # 【bug 修复】原来写的是：
        #     for i, existing in enumerate(merged):
        #         if existing.id == message.id:
        #             merged[i] = message
        #             break
        #         else:
        #             merged.append(message)
        # 这里的 else 缩进和 if 对齐，是 if/else，而不是 for/else！
        # 意味着"只要当前这个 existing 不匹配"就会 append 一次，导致同一条 message
        # 在遍历 merged 的过程中被重复 append 很多次（merged 有几个不匹配的元素就 append 几次）。
        # 正确写法应该是 for/else：把 else 和 for 对齐，只有整个 for 循环
        # "从头跑到尾都没有 break"（也就是没找到相同 id 的旧消息）才 append 一次。
        for i, existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]  # 自定义 reducer：按消息 id 去重/替换，而不是单纯累加

In [ ]:
tool = TavilySearchResults(max_results=2)

## Manual human approval

In [ ]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]   # 关键：在真正执行 "action" 节点（调用工具）之前暂停，把控制权交还给人
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)   # 打印出来方便观察每次判断时 state 里到底有什么
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-3.5-turbo")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
# 因为编译时设置了 interrupt_before=["action"]，模型一旦决定要调用搜索工具，图就会在执行 action 节点之前暂停并返回，
# 不会真的去调用工具——这时可以先检查一下模型想调用什么工具、传了什么参数，再决定要不要放行

In [ ]:
abot.graph.get_state(thread)  # 查看这个 thread 当前保存的完整 state 快照（StateSnapshot），包括消息历史和下一步要跑的节点

In [ ]:
abot.graph.get_state(thread).next  # .next 是接下来要执行的节点名元组，应该是 ('action',)，证实图确实暂停在 action 之前

### continue after interrupt

In [ ]:
for event in abot.graph.stream(None, thread):   # 传入 input=None：不新增消息，只是告诉图"继续跑之前暂停的地方"
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)  # 图跑完之后再看 state，消息历史应该已经包含了工具调用结果和模型的最终回答

In [ ]:
abot.graph.get_state(thread).next  # 图已经跑到 END，这里应该是空元组 ()，表示没有下一步了

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
# 这里演示一种交互式的人工审批循环：只要图还没跑到终点（.next 非空），
# 就打印当前 state，询问用户 "proceed?"，输入 y 才继续 stream(None, thread)，否则直接中止
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}   # 换一个新 thread_id，接下来演示"修改模型想调用的工具参数"这种人工干预方式
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)  # 图应该又暂停在 action 之前，state 里最后一条消息带着模型想执行的 tool_calls

In [ ]:
current_values = abot.graph.get_state(thread)  # 把当前快照存下来，接下来要在这个快照上直接修改内容

In [ ]:
current_values.values['messages'][-1]

In [ ]:
current_values.values['messages'][-1].tool_calls

In [ ]:
# 人工修改模型请求的搜索关键词：把 "current weather in LA" 偷偷改成 "current weather in Louisiana"，
# 演示"人工审核并纠正 Agent 想执行的操作"这一核心场景（保留原来的 tool_call_id 不变，否则模型对不上号）
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

In [ ]:
abot.graph.update_state(thread, current_values.values)  # 把修改后的 state 写回 checkpointer，替换掉暂停时保存的那份

In [ ]:
abot.graph.get_state(thread)  # 确认 state 里的 tool_calls 已经变成修改后的 "Louisiana" 查询

In [ ]:
for event in abot.graph.stream(None, thread):   # 继续执行，这次 action 节点会真的用修改后的参数去调用工具
    for v in event.values():
        print(v)

In [ ]:
states = []
for state in abot.graph.get_state_history(thread):   # 遍历这个 thread 从头到尾保存过的所有历史快照（越新的越先返回）
    print(state)
    print('--')
    states.append(state)

In [ ]:
to_replay = states[-3]  # 取倒数第三个历史快照（也就是较早期的某个 state），准备从这里"重放"

In [ ]:
to_replay  # 看看这个历史快照长什么样：包含 values（state 内容）和 config（可以用来定位/恢复到这一步的配置）

In [ ]:
for event in abot.graph.stream(None, to_replay.config):   # 把历史快照的 config 当成 thread 传进去，从那一步原样重新执行一遍
    for k, v in event.items():
        print(v)

In [ ]:
to_replay  # 再看一次这个历史快照，接下来要在这个"过去的分叉点"上做修改

In [ ]:
# 修改这个历史快照里模型想执行的搜索关键词，等会要基于这个"被修改过的过去"重新分叉出一条新的执行路径
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [ ]:
# 关键点：update_state 传入的是"历史快照的 config"（而不是当前 thread），
# LangGraph 会基于这个历史点创建一个新的分支 config，不会覆盖 thread 主线上更晚的历史
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

In [ ]:
for event in abot.graph.stream(None, branch_state):   # 从这个新分支的 config 继续跑，走出一条和主线不同的执行路径
    for k, v in event.items():
        if k != "__end__":
            print(v)

In [ ]:
to_replay  # 最后演示第三种人工干预方式：不改 tool_calls 的参数，而是直接"伪造"一个工具执行结果注入进去

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']  # 拿到原始 tool_call 的 id，伪造的 ToolMessage 要用同一个 id 才能对上号

In [ ]:
# 手动构造一条 ToolMessage，假装工具已经执行完并返回了这个结果——完全跳过真正调用搜索工具这一步
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search_results_json",
    content="54 degree celcius",
)]}

In [ ]:
# as_node="action"：告诉 LangGraph"这次状态更新就当作是 action 节点刚刚执行完产生的"，
# 这样下一步会正确地回到 "llm" 节点（走 action -> llm 这条边），而不是把它当成普通的外部更新
branch_and_add = abot.graph.update_state(
    to_replay.config,
    state_update,
    as_node="action")

In [ ]:
for event in abot.graph.stream(None, branch_and_add):   # 继续跑：模型会直接拿着这个伪造的 "54 degree celcius" 结果去生成最终回答
    for k, v in event.items():
        print(v)

In [ ]:
# ==== 下面开始是一个不涉及 LLM 的简化 demo：node1 -> node2 循环计数，专门用来演示 state history / 时间旅行 ====
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.sqlite import SqliteSaver  # 后面这个小 demo 图同样需要 checkpointer 才能记录 state 历史

## Time Travel（时间旅行）

下面换成一个更简单的 `node1 -> node2 -> (循环或结束)` 计数器图，不涉及 LLM 调用，专门用来更清楚地演示
`get_state_history` 时间旅行：拿到过去任意一步的快照 `config`，可以直接从那一步重新执行，
甚至可以先手动改一下那一步的 state 再执行，从而"改写历史、分叉出新的执行路径"。

In [ ]:
class AgentState(TypedDict):
    lnode: str          # 记录最近一次是哪个节点执行的（用于观察 state 变化）
    scratch: str         # 一个普通字段，没有自定义 reducer，默认会被直接覆盖（而不是累加）
    count: Annotated[int, operator.add]   # 用 operator.add 让每个节点返回的 count 增量累加起来

In [ ]:
def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,        # 返回 1，会被 operator.add 累加进 state 现有的 count
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

In [ ]:
def should_continue(state):
    return state["count"] < 3   # 条件边：count 还没到 3 就继续回 Node1，否则结束

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")            # 固定边：Node1 跑完永远接着跑 Node2
builder.add_conditional_edges("Node2",
                              should_continue,
                              {True: "Node1", False: END})   # 条件边：Node2 跑完根据 count 决定是回到 Node1 还是结束
builder.set_entry_point("Node1")

In [ ]:
# 【版本变化修复】同样需要用 .__enter__() 拿到真正的 SqliteSaver 实例，并保留上下文管理器引用避免被垃圾回收关闭连接
_memory_cm2 = SqliteSaver.from_conn_string(":memory:")
memory = _memory_cm2.__enter__()
graph = builder.compile(checkpointer=memory)

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
graph.invoke({"count":0, "scratch":"hi"},thread)   # 起始 state：count=0，图会在 Node1/Node2 之间循环直到 count>=3

In [ ]:
graph.get_state(thread)  # 图跑到 END 了，.next 应该是空元组

In [ ]:
for state in graph.get_state_history(thread):   # 完整打印每一步的历史快照，count 应该是 4,3,2,1,0,0（含初始状态）
    print(state, "\n")

In [ ]:
states = []
for state in graph.get_state_history(thread):
    states.append(state.config)     # 这次只保留每一步的 config（后面用它来定位/恢复到某一步）
    print(state.config, state.values['count'])

In [ ]:
states[-3]  # 取倒数第三步的 config，对应 count=1 那个时间点

In [ ]:
graph.get_state(states[-3])  # 可以直接用某个历史 config 查询那一刻的完整 state 快照

In [ ]:
graph.invoke(None, states[-3])   # 时间旅行：从 count=1 这个历史点重新往下跑，而不是从最新状态继续

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):   # 注意：从历史点重新 invoke 之后，thread 的最新历史已经被这次重放覆盖/追加了
    print(state.config, state.values['count'])

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):   # 完整打印，对比重放前后历史链条的变化
    print(state,"\n")

In [ ]:
thread2 = {"configurable": {"thread_id": str(2)}}   # 换一个全新的 thread_id，接下来演示"修改历史 state 的值再重放"
graph.invoke({"count":0, "scratch":"hi"},thread2)

In [ ]:
from IPython.display import Image
# 【运行环境问题修复】同 Lesson 2：draw_png() 依赖本地未安装的 pygraphviz，会直接 ImportError，
# 这里改用不需要额外依赖、纯本地生成文本的 draw_mermaid()（把输出贴到 Mermaid 编辑器里即可看到图）
print(graph.get_graph().draw_mermaid())

In [ ]:
states2 = []
for state in graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values['count'])

In [ ]:
save_state = graph.get_state(states2[-3])   # 取出某个历史时刻的完整快照，准备直接修改它的内容（而不是修改 tool_calls）
save_state

In [ ]:
save_state.values["count"] = -3        # 直接把 count 改成一个"不可能自然产生"的值，方便等下一眼认出这是被人工修改过的分支
save_state.values["scratch"] = "hello"
save_state

In [ ]:
# 注意这里传的是 thread2（当前/最新的 thread），不是 save_state 所在的历史 config，
# 所以这次更新会作用在 thread2 的最新状态上（相当于直接把最新 state 的 count/scratch 改掉），而不是分叉出新历史
graph.update_state(thread2,save_state.values)

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

In [ ]:
# as_node="Node1"：告诉 LangGraph 把这次更新当成是 Node1 刚执行完产生的结果，
# 这样 .next 会正确地指向 Node1 之后该走的节点（Node2），而不是把更新当成孤立的手动补丁
graph.update_state(thread2,save_state.values, as_node="Node1")

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

In [ ]:
graph.invoke(None,thread2)  # 继续跑 thread2：会从被人工改成 count=-3 的这个"伪造起点"开始，重新循环到 count>=3 为止

In [ ]:
for state in graph.get_state_history(thread2):   # 最终完整历史：可以看到 count 是从人为设置的 -3 开始重新往上累加的
    print(state,"\n")